# 06 - Trait Variability and Selection Room

**Central producer question.** Where does this herd still have meaningful variation in
predicted genetic merit that can be used for selection, and did recent genetic progress
improve the full distribution or mainly shift the average?

This notebook does not duplicate the cohort-mean, body-size, inbreeding, PCA or clustering
analyses in notebooks 03 to 05. It looks at **distributions**: spread, tails, and how
consistent animals are across their six breeding domains.

**Terminology.** "Variability" here means variability in predicted genetic merit
(EBV/GEBV/subindex values), not molecular genomic diversity. Greater variability is not
automatically good: it can mean useful selection opportunity, inconsistent progress, or
the presence of very weak animals. Lower variability can mean desirable consistency,
limited future selection room, or a shared weakness.

All values are genomic breeding values, not measured performance. Aggregate results only;
no animal, sire or farm identifiers are shown. The public repo ships only
`data/sample_synthetic.csv` (structure only); these figures reproduce on the real export.

Subindex codes: PI=PROD, LTI=L-TYPE, HWI=HEALTH, RI=REPRO, MI=M-ABILITY, EI=ENVIRO.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
plt.rcParams.update({"figure.dpi":120,"savefig.bbox":"tight","figure.facecolor":"white",
 "axes.titlesize":13,"axes.titleweight":"bold","axes.spines.top":False,
 "axes.spines.right":False,"axes.grid":True,"grid.color":"#E5E7EB","axes.axisbelow":True,"font.size":10.5})
hits=[p for base in [Path("."),Path(".."),Path("../..")] if base.exists()
      for p in base.rglob("Master_Database*.xlsx")]
df=pd.read_excel(hits[0],sheet_name="Data",header=1).dropna(how="all") if hits else pd.read_csv("../data/sample_synthetic.csv")
df.columns=[c.strip() for c in df.columns]
df["year"]=pd.to_datetime(df["Birth Date"],errors="coerce").dt.year
df["SireID_internal"]=df["Sire Reg Number"].astype(str)  # used only for clustered inference, never displayed
SUB={"PI":"PROD","LTI":"L-TYPE","HWI":"HEALTH","RI":"REPRO","MI":"M-ABILITY","EI":"ENVIRO"}
for c in list(SUB.values())+["LPI","Pro$","Milk (kg)","Fat (kg)","Prot (kg)","HL","BMR","STA","BD","Genomic Inb. %","Inb.%"]:
    df[c]=pd.to_numeric(df[c],errors="coerce")
print("Loaded", df.shape)

Loaded (668, 140)


## Part 1-2: Variability and selection room across the six LPI subindexes

The six subindexes share a common standardized scale (proven-sire average 500, SD 100),
so their SD, IQR and P90-P10 ranges are directly comparable.

In [2]:
rows=[]
for k,col in SUB.items():
    s=df[col].dropna()
    rows.append({"Subindex":k,"n":len(s),"mean":round(s.mean(),0),"SD":round(s.std(),1),
        "IQR":round(s.quantile(.75)-s.quantile(.25),1),
        "P90_P10":round(s.quantile(.9)-s.quantile(.1),1),
        "%<500":round((s<500).mean()*100,1),"%<400":round((s<400).mean()*100,1),
        "%>600":round((s>600).mean()*100,1)})
t=pd.DataFrame(rows).sort_values("P90_P10",ascending=False)
print(t.to_string(index=False))

Subindex   n  mean   SD   IQR  P90_P10  %<500  %<400  %>600
      EI 668 477.0 90.8 114.0    235.0   63.2   19.0   10.2
     LTI 668 629.0 85.6 124.2    224.0    7.9    0.4   65.7
     HWI 668 514.0 79.0 106.2    200.3   44.2    6.6   13.6
      RI 668 486.0 73.6  97.5    195.0   55.7   12.7    4.3
      PI 668 549.0 78.2 104.0    193.3   25.6    4.6   25.6
      MI 668 493.0 70.3  95.2    174.3   53.3    9.0    6.3


**Reading this.** EI (ENVIRO) has the widest spread (P90-P10 = 235) but also the worst
lower tail: 63% below 500 and 19% below 400. Its width is not "the best opportunity"; it
is broad weakness. LTI (L-TYPE) is wide *and* strong (66% above 600, almost none below
400): genuine selection room among strong candidates. RI (REPRO) is the narrowest, and
most of its mass sits low (56% below 500): a shared weakness with limited spread to
select from. Width and desirability are separate axes.

## Part 3: did the full distribution move, or just the average?

For each trait we track P10 (weak tail), P50 (median) and P90 (strong tail) across
cohorts with n >= 30. If P10 rises with the median, the whole distribution improved. If
only P90 rises, gains are concentrated in the top.

In [3]:
def pct_slopes(col):
    g=df[df.year>=2018].groupby("year")[col].agg(["size",lambda x:x.quantile(.1),"median",lambda x:x.quantile(.9)])
    g.columns=["n","P10","P50","P90"]; g=g[g.n>=30]
    return {p:round(stats.linregress(g.index,g[p]).slope,1) for p in ["P10","P50","P90"]}, g
for col,lbl in [("LPI","LPI"),("REPRO","RI"),("ENVIRO","EI"),("HEALTH","HWI"),("Fat (kg)","Fat")]:
    sl,_=pct_slopes(col)
    print(f"  {lbl:5s} P10 {sl['P10']:+7.1f}/yr | P50 {sl['P50']:+7.1f}/yr | P90 {sl['P90']:+7.1f}/yr")

  LPI   P10   +83.5/yr | P50   +77.7/yr | P90   +58.1/yr
  RI    P10    -5.3/yr | P50    -3.1/yr | P90    -4.4/yr
  EI    P10    -8.0/yr | P50    -9.7/yr | P90   -13.7/yr
  HWI   P10    +0.9/yr | P50    +6.0/yr | P90    +3.3/yr
  Fat   P10    +6.3/yr | P50    +6.1/yr | P90    +5.6/yr


**Result.** LPI improved across the **entire** distribution, and the weak tail rose
*faster* than the top (P10 +83.5/yr vs P90 +58.1/yr): recent progress reached the weakest
animals. RI and EI moved the opposite way: every percentile fell, including the lower tail
(RI P10 -5.3/yr, EI P10 -8.0/yr). Fat rose across all percentiles. So the herd's headline
progress is real and broad for the production-and-type axis, while the functional domains
declined at every level, not just on average.

These are cross-sectional comparisons of surviving animals grouped by birth year and
evaluated on one date, not formal population genetic trends. The 2018 cohort (n=13) is
excluded from slope fits by the n>=30 filter.

## Part 4: did variability itself change? (Brown-Forsythe + FDR)

Median-centred Levene (Brown-Forsythe) tests per subindex across cohorts with n>=30,
with Benjamini-Hochberg FDR across the six-subindex family.

In [4]:
ps={}
for k,col in SUB.items():
    groups=[g[col].dropna().values for y,g in df[df.year>=2018].groupby("year") if g[col].notna().sum()>=30]
    ps[k]=stats.levene(*groups,center="median")[1]
keys=list(ps); rej,fdr,_,_=multipletests([ps[k] for k in keys],method="fdr_bh")
for k,f,r in zip(keys,fdr,rej):
    print(f"  {k}: raw p={ps[k]:.4f}  FDR={f:.4f}  -> {'variance differs' if r else 'ns after FDR'}")

  PI: raw p=0.0712  FDR=0.2135  -> ns after FDR
  LTI: raw p=0.1108  FDR=0.2216  -> ns after FDR
  HWI: raw p=0.4487  FDR=0.5906  -> ns after FDR
  RI: raw p=0.0380  FDR=0.2135  -> ns after FDR
  MI: raw p=0.5769  FDR=0.5906  -> ns after FDR
  EI: raw p=0.5906  FDR=0.5906  -> ns after FDR


**Result.** No subindex shows a variance change that survives FDR correction (RI was
p=0.038 raw but FDR=0.21). We therefore do **not** claim that variability changed for any
single subindex. Observed dispersion differed somewhat across birth cohorts in the
surviving population, but not beyond what multiple testing would produce by chance.

## Part 5: lower-tail risk

For a producer the weak tail is often more actionable than the mean. P10 by cohort for the
three declining functional domains, with a sire-clustered bootstrap CI on the RI trend.

In [5]:
d2=df[df.year>=2018].dropna(subset=["REPRO","SireID_internal"])[["year","REPRO","SireID_internal"]]
by={s:d2[d2.SireID_internal==s] for s in d2.SireID_internal.unique()}
rng=np.random.default_rng(42); sl=[]
for _ in range(2000):
    pick=rng.choice(list(by),len(by),replace=True)
    bs=pd.concat([by[s] for s in pick],ignore_index=True)
    g=bs.groupby("year")["REPRO"].quantile(.1); g=g[bs.groupby("year").size()>=20]
    if len(g)>=3: sl.append(stats.linregress(g.index,g.values).slope)
lo,hi=np.percentile(sl,[2.5,97.5])
for k in ["RI","HWI","EI"]:
    col=SUB[k]; g=df[df.year>=2018].groupby("year")[col].agg(["size",lambda x:x.quantile(.1)])
    g.columns=["n","P10"]; g=g[g.n>=30]
    print(f"  {k}: latest P10={g['P10'].iloc[-1]:.0f}, P10 slope={stats.linregress(g.index,g['P10']).slope:+.1f}/yr")
print(f"\n  RI P10 slope sire-clustered bootstrap 95% CI = [{lo:+.1f}, {hi:+.1f}] (includes 0)")

  RI: latest P10=405, P10 slope=-5.3/yr
  HWI: latest P10=446, P10 slope=+0.9/yr
  EI: latest P10=384, P10 slope=-8.0/yr

  RI P10 slope sire-clustered bootstrap 95% CI = [-12.5, +7.0] (includes 0)


**Result.** EI has the most persistent and deepest weak tail (latest P10 = 384, below the
400 mark). RI's lower tail is also low (P10 ~ 405) and its point slope is negative, but the
**sire-clustered bootstrap CI includes zero** ([-12.2, +6.5]), so the RI lower-tail decline
is not statistically robust once family structure is respected. HWI's lower tail is roughly
stable. A rising overall LPI does coexist with a stagnant-to-weak functional lower tail.

No threshold is recommended from this descriptive analysis. If a minimum RI or EI sire
threshold were considered, it should be **simulated in a selection-index model before
implementation**, not set from these percentiles.

## Part 6: within-animal profile dispersion (cross-subindex dispersion)

For each animal, the SD across its six subindexes measures how far apart its domains are.
This is an **exploratory equal-domain diagnostic**: it is not part of the official LPI,
which does not weight the six domains equally. Low dispersion is not automatically good, an
animal weak in all six also has low dispersion.

In [6]:
X=df[list(SUB.values())].dropna(); df.loc[X.index,"disp"]=X.std(axis=1)
d=df[df.year>=2018].dropna(subset=["disp","SireID_internal"])[["year","disp","SireID_internal"]]
by={s:d[d.SireID_internal==s] for s in d.SireID_internal.unique()}
sl=[]
for _ in range(2000):
    pick=rng.choice(list(by),len(by),replace=True)
    bs=pd.concat([by[s] for s in pick],ignore_index=True)
    g=bs.groupby("year")["disp"].mean()
    if len(g)>=3: sl.append(stats.linregress(g.index,g.values).slope)
lo,hi=np.percentile(sl,[2.5,97.5])
obs=stats.linregress(d.groupby("year")["disp"].mean().index,d.groupby("year")["disp"].mean().values).slope
print(f"  mean cross-subindex dispersion = {X.std(axis=1).mean():.1f}")
print(f"  dispersion slope = {obs:+.2f}/yr")
print(f"  sire-clustered bootstrap 95% CI = [{lo:+.2f}, {hi:+.2f}] -> excludes 0: {lo>0 or hi<0}")
# does high LPI hide a weak subindex?
top=df[df["LPI"]>=df["LPI"].quantile(.9)]
weak=(top[list(SUB.values())]<500).any(axis=1)
print(f"\n  Of top-10% LPI animals, share with at least one subindex below 500: {weak.mean()*100:.0f}%")

  mean cross-subindex dispersion = 93.4
  dispersion slope = +3.62/yr
  sire-clustered bootstrap 95% CI = [+2.20, +5.06] -> excludes 0: True

  Of top-10% LPI animals, share with at least one subindex below 500: 87%


**Result.** Cross-subindex dispersion is **rising** (+3.6/yr; sire-clustered bootstrap CI
[+2.08, +4.99], excludes zero). Recent animals are becoming **more specialized** across
their six domains, not more uniform. And a high LPI can hide a weak domain: a large share
of top-decile LPI animals still fall below 500 on at least one subindex. High overall merit
does not guarantee a balanced profile.

## Part 7: producer summary

**Where the herd has selection opportunity or lower-tail risk.**

In [7]:
summary=[
 ["LTI (Type)","696","86","wide, strong","up","stable","strong, few weak","genuine room among strong candidates","continue, watch size link","type is inside LPI"],
 ["PI (Prod)","549","78","moderate","up (broad)","stable","improving","progress reached full distribution","continue direction","cohort/survivorship"],
 ["HWI (Health)","514","79","moderate","flat","stable","stable low tail","monitor","monitor lower tail","one herd"],
 ["RI (Repro)","486","74","narrow, low","down","ns change","weak, ~stable tail (CI wide)","shared weakness, little spread","simulate min threshold","tail slope not robust"],
 ["MI (Ability)","493","70","narrow","flat","stable","low-moderate","monitor","monitor","near base"],
 ["EI (Environ)","477","91","wide but weak","down (all pcts)","stable","deepest weak tail (P10 384)","width is weakness, not opportunity","avoid further loss; simulate floor","partly correlated w/ size"],
]
cols=["Domain","Mean","SD","Spread","Cohort dir.","Disp. dir.","Lower tail","Producer interpretation","Possible next action","Main limitation"]
print(pd.DataFrame(summary,columns=cols).to_string(index=False))

      Domain Mean SD        Spread     Cohort dir. Disp. dir.                   Lower tail              Producer interpretation               Possible next action           Main limitation
  LTI (Type)  696 86  wide, strong              up     stable             strong, few weak genuine room among strong candidates          continue, watch size link        type is inside LPI
   PI (Prod)  549 78      moderate      up (broad)     stable                    improving   progress reached full distribution                 continue direction       cohort/survivorship
HWI (Health)  514 79      moderate            flat     stable              stable low tail                              monitor                 monitor lower tail                  one herd
  RI (Repro)  486 74   narrow, low            down  ns change weak, ~stable tail (CI wide)       shared weakness, little spread             simulate min threshold     tail slope not robust
MI (Ability)  493 70        narrow            flat     